In [ ]:
import numpy as np
import pandas as pd

from src.datasets.sms_spam_collection import _load, _split, _preprocess

In [ ]:
df = _load()
train_df, val_df, test_df = _split(df, val_size=0.2, test_size=0.1, random_state=42)

print(f'full: {df.shape}')
print(f'train: {train_df.shape}, val: {val_df.shape}, test: {test_df.shape}')
display(train_df.head())

In [ ]:
data = _preprocess(train_df, val_df, test_df, mode='tfidf', max_features=15000)

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val:  ', X_val.shape, 'y_val:  ', y_val.shape)
print('X_test: ', X_test.shape, 'y_test: ', y_test.shape)

In [ ]:
# Примитивная проверка дисбаланса классов
train_spam_rate = y_train.mean()
val_spam_rate = y_val.mean()
test_spam_rate = y_test.mean()

print(f'spam rate train: {train_spam_rate:.3f}')
print(f'spam rate val:   {val_spam_rate:.3f}')
print(f'spam rate test:  {test_spam_rate:.3f}')

In [ ]:
from src.model import (
    LogisticRegressionClassifier,
    MultinomialNBClassifier,
    BernoulliNBClassifier,
    SVCClassifier,
    RandomForestClassifierModel,
    KNeighborsClassifierModel,
    DecisionTreeClassifierModel,
)


In [ ]:
models = {
    'logreg': LogisticRegressionClassifier(max_iter=1000),
    'multinomial_nb': MultinomialNBClassifier(alpha=1.0),
    'bernoulli_nb': BernoulliNBClassifier(alpha=1.0),
    'svc_rbf': SVCClassifier(C=1.0, kernel='rbf', gamma='scale'),
    'random_forest': RandomForestClassifierModel(n_estimators=300, random_state=42, n_jobs=-1),
    'knn': KNeighborsClassifierModel(n_neighbors=5),
    'decision_tree': DecisionTreeClassifierModel(random_state=42),
}

rows = []
failed = []
for name, model in models.items():
    try:
        model.fit(X_train, y_train)
        val_report = model.evaluate(X_val, y_val)
        test_report = model.evaluate(X_test, y_test)

        rows.append({
            'model': name,
            'split': 'val',
            **val_report['metrics'],
        })
        rows.append({
            'model': name,
            'split': 'test',
            **test_report['metrics'],
        })
    except Exception as exc:
        failed.append({'model': name, 'error': str(exc)})

results = pd.DataFrame(rows).sort_values(['split', 'f1'], ascending=[True, False])
display(results.round(4))

if failed:
    print('failed models:')
    display(pd.DataFrame(failed))

best_row = results[results['split'] == 'val'].sort_values('f1', ascending=False).iloc[0]
print(f"best by val f1: {best_row['model']} ({best_row['f1']:.4f})")
